#### Importação de bibliotecas

<small>

| Biblioteca | Uso |
|---|---|
| `os` | caminhos e arquivos do sistema |
| `cv2` | leitura e conversão de imagens (BGR → RGB, grayscale) |
| `numpy` | arrays numéricos e normalização |
| `torch` | framework de deep learning — tensores, gradientes, GPU |
| `torch.utils.data` | estrutura de dataset customizado e batches |
| `albumentations` | pipeline de augmentação sincronizada entre imagem e máscara |
| `albumentations.pytorch` | conversão final de arrays numpy para tensores PyTorch |
| `matplotlib.pyplot` | visualização de imagens e métricas |

<small>

In [1]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

#### Dataset LEVIR-CD

Classe que carrega cada amostra como um triplo `(imagem_pré, imagem_pós, máscara)`, aplica normalização e augmentações sincronizadas entre as duas imagens e a máscara binária.

In [2]:
class LEVIRCDDataset(Dataset):
    def __init__(self, images_dir, transform=None):
        # caminhos para imagens pré-mudança (A), pós-mudança (B) e máscaras
        self.images_dir_a = os.path.join(images_dir, 'A')  # antes
        self.images_dir_b = os.path.join(images_dir, 'B')  # depois
        self.masks_dir = os.path.join(images_dir, 'label') # máscara
        self.transform = transform

        # lista ordenada de arquivos de imagem em cada diretório
        self.images_a = sorted(f for f in os.listdir(self.images_dir_a) 
                               if f.endswith(('.png', '.jpg', '.jpeg')))
        self.images_b = sorted(f for f in os.listdir(self.images_dir_b) 
                               if f.endswith(('.png', '.jpg', '.jpeg')))
        self.masks = sorted(f for f in os.listdir(self.masks_dir) 
                            if f.endswith(('.png', '.jpg', '.jpeg')))

    def __len__(self):
        return len(self.images_a)

    def __getitem__(self, idx):
        # monta caminhos completos para o par de imagens e a máscara correspondente
        img_path_a = os.path.join(self.images_dir_a, self.images_a[idx])
        img_path_b = os.path.join(self.images_dir_b, self.images_b[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])
                
        # leitura e pré-processamento das imagens: BGR → RGB → float normalizado [0, 1]
        image_a = cv2.imread(img_path_a, cv2.IMREAD_COLOR)
        image_a = cv2.cvtColor(image_a, cv2.COLOR_BGR2RGB)
        image_a = image_a.astype(np.float32) / 255.0

        image_b = cv2.imread(img_path_b, cv2.IMREAD_COLOR)
        image_b = cv2.cvtColor(image_b, cv2.COLOR_BGR2RGB)
        image_b = image_b.astype(np.float32) / 255.0

        # leitura da máscara binária em escala de cinza e normalização para [0, 1]        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = mask.astype(np.float32) / 255.0

        # aplica augmentações sincronizadas nas duas imagens e na máscara
        if self.transform:
            augmented = self.transform(image=image_a, image1=image_b, mask=mask)
            image_a = augmented['image']
            image_b = augmented['image1']
            mask = augmented['mask']
                
        # adiciona dimensão de canal à máscara: (H, W) → (1, H, W)
        mask = mask.unsqueeze(0)

        return image_a, image_b, mask

# transformações albumentations
# treino: redimensionamento + augmentações geométricas aleatórias + conversão para tensor
train_transform = A.Compose([
    A.Resize(height=256, width=256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    ToTensorV2(),
], additional_targets={'image1': 'image'})  # sincroniza augmentação entre imagem pré e pós

# validação: apenas redimensionamento e conversão para tensor (sem augmentação)
val_transform = A.Compose([
    A.Resize(height=256, width=256),
    ToTensorV2(),
], additional_targets={'image1': 'image'})


#### Criação dos datasets e DataLoaders

Instancia os três splits do LEVIR-CD (treino, validação, teste) com seus respectivos transforms, com augmentação apenas no treino, e organiza os dados em batches de tamanho `BATCH_SIZE`.

In [3]:
# caminho raiz do dataset LEVIR-CD e tamanho do batch
root = os.path.join(os.getcwd(), "LEVIR_CD")
batch_size = 16

LEVIR_train_dataset = LEVIRCDDataset(
    images_dir = os.path.join(root, 'train'),
    transform = train_transform
)

LEVIR_val_dataset = LEVIRCDDataset(
    images_dir = os.path.join(root, 'val'),
    transform = val_transform
)

LEVIR_test_dataset = LEVIRCDDataset(
    images_dir = os.path.join(root, 'test'),
    transform = val_transform # sem augmentação, igual à validação
)

# criação dos DataLoaders para treino, validação e teste
# shuffle = True no treino para embaralhar os batches e melhorar a generalização
LEVIR_train_loader = DataLoader(LEVIR_train_dataset, batch_size=batch_size, shuffle=True)
LEVIR_val_loader = DataLoader(LEVIR_val_dataset, batch_size=batch_size, shuffle=False)
LEVIR_test_loader = DataLoader(LEVIR_test_dataset, batch_size=batch_size, shuffle=False)

#### Verificação da divisão do dataset

Confirma o número de amostras carregadas em cada split (treino, validação, teste).

In [ ]:
print(f'Treino:     {len(LEVIR_train_dataset)} amostras')
print(f'Validação:  {len(LEVIR_val_dataset)} amostras')
print(f'Teste:      {len(LEVIR_test_dataset)} amostras')

#### Sanidade do dataset

Verifica shapes, valores e normalização de uma amostra antes de iniciar o treinamento.

In [ ]:
image_pre, image_post, mask = LEVIR_train_dataset[10]

print("── Shapes ──────────────────────────────")
print(f"  Imagem pré : {tuple(image_pre.shape)}   esperado: (3, 256, 256)")
print(f"  Imagem pós : {tuple(image_post.shape)}   esperado: (3, 256, 256)")
print(f"  Máscara    : {tuple(mask.shape)}   esperado: (1, 256, 256)")

print("── Normalização ────────────────────────")
print(f"  Imagem pré : min={image_pre.min():.3f}  max={image_pre.max():.3f}   esperado: 0.0 ~ 1.0")
print(f"  Imagem pós : min={image_post.min():.3f}  max={image_post.max():.3f}   esperado: 0.0 ~ 1.0")

print("── Máscara ─────────────────────────────")
print(f"  Valores únicos : {torch.unique(mask).tolist()}   esperado: [0.0, 1.0]")
print(f"  Classes        : {torch.unique(mask).numel()}   esperado: 2")

#### Visualização de amostras do dataset

Exibe amostras `(pré, pós, máscara)` do conjunto de treino para confirmar alinhamento, qualidade das máscaras e resultado das augmentações.

In [ ]:
# visualização de pares de imagens e máscaras do dataset de treino
fig, axes = plt.subplots(3, 3, figsize = (15, 10))
fig.suptitle('Amostras do dataset treino', fontsize = 14)

for i in range(3):
    image1, image2, mask = LEVIR_train_dataset[i]

    # pré-mudança (A): tensor (C, H, W) → (H, W, C) para o matplotlib
    axes[i, 0].imshow(image1.permute(1, 2, 0))
    axes[i, 0].set_title(f'Pré-mudança {i+1}')
    axes[i, 0].axis('off')

    # pós-mudança (B)
    axes[i, 1].imshow(image2.permute(1, 2, 0))
    axes[i, 1].set_title(f'Pós-mudança {i+1}')
    axes[i, 1].axis('off')

    # máscara binária: áreas amarelas = mudança detectada
    axes[i, 2].imshow(mask.permute(1, 2, 0), cmap='viridis')
    axes[i, 2].set_title(f'Máscara {i+1}')
    axes[i, 2].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

#### Arquitetura da Siamese U-Net

O modelo processa o par de imagens (pré e pós-mudança) em dois encoders independentes. A diferença entre os features de cada encoder indica onde houve mudança, e o decoder usa essa informação para reconstruir a máscara final.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# bloco de construção base: Convolução → BatchNorm → ReLU
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1) # padding=1 mantém H e W
        self.bn = nn.BatchNorm2d(out_channels) # estabiliza e acelera o treinamento
        self.relu = nn.ReLU(inplace=True) # inplace economiza memória

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

# bloco do encoder: dois ConvBlocks + MaxPooling para reduzir resolução
class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.conv1 = ConvBlock(in_channels, out_channels)
        self.conv2 = ConvBlock(out_channels, out_channels)
        self.pool = nn.MaxPool2d(2) # reduz H e W pela metade

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        p = self.pool(x)
        return x, p  # x = skip connection para o decoder; p = entrada para o próximo encoder

# bloco do decoder: upsampling + concatenação com skip + dois ConvBlocks
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.upconv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv1 = ConvBlock(out_channels + skip_channels, out_channels)        
        self.conv2 = ConvBlock(out_channels, out_channels)

    def forward(self, x, skip):
        x = self.upconv(x)
        x = torch.cat([x, skip], dim=1)  # concatena ao longo dos canais
        x = self.conv1(x)
        x = self.conv2(x)
        return x

# Siamese U-Net: dois encoders independentes (um por imagem) + decoder compartilhado
# a diferença entre features dos dois encoders serve como skip connection,
# destacando regiões de mudança entre pré e pós.
class SiameseUNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(SiameseUNet, self).__init__()

        # encoder da imagem pré-mudança (A)
        self.encoder1a = EncoderBlock(in_channels, 64)
        self.encoder2a = EncoderBlock(64, 128)
        self.encoder3a = EncoderBlock(128, 256)
        self.encoder4a = EncoderBlock(256, 512)

        # encoder da imagem pós-mudança (B) — mesma estrutura, pesos independentes
        self.encoder1b = EncoderBlock(in_channels, 64)
        self.encoder2b = EncoderBlock(64, 128)
        self.encoder3b = EncoderBlock(128, 256)
        self.encoder4b = EncoderBlock(256, 512)

        # bottleneck: recebe a concatenação dos dois encoders (512 + 512 = 1024 canais)
        self.bottleneck = ConvBlock(1024, 1024)  # 512 + 512 from both encoders

        # decoder compartilhado: reconstrói a resolução original progressivamente
        self.decoder4 = DecoderBlock(1024, 512, 512)
        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)

        # convolução final 1×1: mapeia 64 canais para o mapa de segmentação (1 canal = binário)
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x1, x2):
        # extração de features da imagem pré-mudança
        e1a, p1a = self.encoder1a(x1)
        e2a, p2a = self.encoder2a(p1a)
        e3a, p3a = self.encoder3a(p2a)
        e4a, p4a = self.encoder4a(p3a)

        # extração de features da imagem pós-mudança
        e1b, p1b = self.encoder1b(x2)
        e2b, p2b = self.encoder2b(p1b)
        e3b, p3b = self.encoder3b(p2b)
        e4b, p4b = self.encoder4b(p3b)

        # diferença absoluta entre features dos dois encoders por nível
        # realça onde houve mudança entre as imagens
        diff_e4 = torch.abs(e4a - e4b)
        diff_e3 = torch.abs(e3a - e3b)
        diff_e2 = torch.abs(e2a - e2b)
        diff_e1 = torch.abs(e1a - e1b)

        # bottleneck: concatena as saídas mais profundas dos dois encoders
        b = self.bottleneck(torch.cat([p4a, p4b], dim=1))

        # decoder: upsampling progressivo usando as diferenças como skip connections
        d4 = self.decoder4(b, diff_e4)
        d3 = self.decoder3(d4, diff_e3)
        d2 = self.decoder2(d3, diff_e2)
        d1 = self.decoder1(d2, diff_e1)

        # saída final: mapa binário de mudança (batch, 1, H, W)
        return self.final_conv(d1)

# verifica se o modelo aceita as dimensões esperadas
x1 = torch.randn(1, 3, 256, 256)  # imagem pré-mudança simulada
x2 = torch.randn(1, 3, 256, 256)  # imagem pós-mudança simulada

siamese_unet = SiameseUNet(in_channels=3, out_channels=1)
output = siamese_unet(x1, x2)

print("Output shape:", output.shape) # esperado: torch.Size([1, 1, 256, 256])

#### Treinamento e validação

A cada epoch o modelo treina no conjunto de treino, avalia no de validação e registra as métricas (Loss, IoU, Dice, Precision, Recall, Accuracy). O melhor modelo é salvo automaticamente com base no IoU de validação.

<small>

| Métrica | Indica |
|---|---|
| `Loss` | erro geral do modelo — **quanto menor, melhor** |
| `IoU` | sobreposição entre máscara predita e real — métrica principal |
| `Dice` | equilíbrio entre precisão e recall — similar ao IoU |
| `Precision` | dos pixels classificados como mudança, quantos realmente eram |
| `Recall` | das mudanças reais, quantas foram detectadas |
| `Accuracy` | pixels corretos no geral — pode ser enganosa em dados desbalanceados |

<small>

In [ ]:
import torch.optim as optim

def calculate_metrics(outputs, labels, threshold=0.5):
    # converte logits em probabilidades e aplica threshold para obter máscara binária
    probs = torch.sigmoid(outputs)
    predictions = (probs > threshold).float()
    
    # achata para vetor 1D para facilitar os cálculos
    predictions = predictions.view(-1)
    labels = labels.float().view(-1)

    # verdadeiros/falsos positivos e negativos
    tp = torch.sum(predictions * labels)
    tn = torch.sum((1 - predictions) * (1 - labels))
    fp = torch.sum(predictions * (1 - labels))
    fn = torch.sum((1 - predictions) * labels)

    # métricas — 1e-6 evita divisão por zero
    precision = tp / (tp + fp + 1e-6)
    recall    = tp / (tp + fn + 1e-6)
    dice      = 2 * tp / (2 * tp + fp + fn + 1e-6)
    iou       = tp / (tp + fp + fn + 1e-6)
    accuracy  = (tp + tn) / (tp + tn + fp + fn + 1e-6)

    return (accuracy.item(), precision.item(), recall.item(), dice.item(), iou.item())

# loop de treinamento e validação
num_epochs = 50

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device='cuda'):
    model.to(device)
    best_iou = 0.0
    # histórico das métricas
    history = {
        'train_loss': [], 'train_iou': [], 'train_acc': [],
        'train_precision': [], 'train_recall': [], 'train_dice': [],
        'val_loss': [],   'val_iou': [],   'val_acc': [],
        'val_precision': [],   'val_recall': [],   'val_dice': [],
    }

    for epoch in range(num_epochs):
        #fase do teste
        model.train()
        running_train_loss = 0.0
        train_iou_sum = train_acc_sum = train_precision_sum = 0
        train_recall_sum = train_dice_sum = train_samples = 0
        
        for inputs1, inputs2, labels in train_loader:
            inputs1, inputs2, labels = inputs1.to(device), inputs2.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs1, inputs2)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # evita explosão de gradientes
            optimizer.step()
            
            # acumula loss e métricas ponderadas pelo tamanho do batch
            n = inputs1.size(0)
            running_train_loss += loss.item() * n
            acc, precision, recall, dice, iou = calculate_metrics(outputs, labels)
            train_acc_sum += acc * n
            train_precision_sum += precision * n
            train_recall_sum += recall * n
            train_dice_sum += dice * n
            train_iou_sum += iou * n
            train_samples += n
        
        # médias ponderadas do epoch de treino
        epoch_train_loss        = running_train_loss    / train_samples
        epoch_train_iou         = train_iou_sum         / train_samples
        epoch_train_acc         = train_acc_sum         / train_samples
        epoch_train_precision   = train_precision_sum   / train_samples
        epoch_train_recall      = train_recall_sum      / train_samples
        epoch_train_dice        = train_dice_sum        / train_samples
        
        # registra no histórico
        history['train_loss'].append(epoch_train_loss)
        history['train_iou'].append(epoch_train_iou)
        history['train_acc'].append(epoch_train_acc)
        history['train_precision'].append(epoch_train_precision)
        history['train_recall'].append(epoch_train_recall)
        history['train_dice'].append(epoch_train_dice)

        # fase de validação
        model.eval()
        running_val_loss = 0.0
        val_iou_sum = val_acc_sum = val_precision_sum   = 0
        val_recall_sum = val_dice_sum = val_samples     = 0
        
        with torch.no_grad(): # desativa gradientes para economizar memória
            for inputs1, inputs2, labels in val_loader:
                inputs1, inputs2, labels = inputs1.to(device), inputs2.to(device), labels.to(device)
                outputs = model(inputs1, inputs2)
                loss = criterion(outputs, labels)
                
                n = inputs1.size(0)
                running_val_loss += loss.item() * n
                acc, precision, recall, dice, iou = calculate_metrics(outputs, labels)
                val_acc_sum += acc * n
                val_precision_sum += precision * n
                val_recall_sum += recall * n                
                val_dice_sum += dice * n
                val_iou_sum += iou * n
                val_samples += n
        
        # médias ponderadas do epoch de validação
        epoch_val_loss      = running_val_loss  / val_samples
        epoch_val_iou       = val_iou_sum       / val_samples
        epoch_val_acc       = val_acc_sum       / val_samples
        epoch_val_precision = val_precision_sum / val_samples
        epoch_val_recall    = val_recall_sum    / val_samples
        epoch_val_dice      = val_dice_sum      / val_samples

        history['val_loss'].append(epoch_val_loss)
        history['val_iou'].append(epoch_val_iou)
        history['val_acc'].append(epoch_val_acc)
        history['val_precision'].append(epoch_val_precision)
        history['val_recall'].append(epoch_val_recall)
        history['val_dice'].append(epoch_val_dice)
        
        # log do epoch
        log = (f'Epoch {epoch+1}/{num_epochs} || '
              f'Treino Loss: {epoch_train_loss:.4f} / Val Loss: {epoch_val_loss:.4f} || '
              f'Treino IoU: {epoch_train_iou:.4f} / Val IoU: {epoch_val_iou:.4f} || '
              f'Treino Accuracy: {epoch_train_acc:.4f} / Val Accuracy: {epoch_val_acc:.4f} || '
              f'Treino Precision: {epoch_train_precision:.4f} / Val Precision: {epoch_val_precision:.4f} || '
              f'Treino Recall: {epoch_train_recall:.4f} / Val Recall: {epoch_val_recall:.4f} || '
              f'Treino Dice: {epoch_train_dice:.4f} / Val Dice: {epoch_val_dice:.4f}')
        
        print(f'--- LOG DE TREINAMENTO ---')
        print(log)
        with open('train_log.txt', 'a') as f:
            f.write(log + '\n')

        # checkpoint ─ salva o melhor modelo com base no IoU de validação
        if epoch_val_iou > best_iou:
            best_iou = epoch_val_iou

            torch.save({
                'epoch':                    epoch + 1,
                'model_state_dict':         model.state_dict(),
                'optimizer_state_dict':     optimizer.state_dict(),
                'iou':                      epoch_val_iou,
                'loss':                     epoch_val_loss
            }, 'best_model.pt')
            print(f' -> best_model.pt salvo! IoU: {best_iou:.6f}')

        # salva checkpoint do último epoch
        torch.save({
            'epoch':                        epoch + 1,
            'model_state_dict':             model.state_dict(),
            'optimizer_state_dict':         optimizer.state_dict(),
            'best_iou':                     best_iou,
            'iou':                          epoch_val_iou,
            'loss':                         epoch_val_loss,
            'train_loss':                   epoch_train_loss
        }, "last_checkpoint.pt")

    return history

# configuração e início do treinamento
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SiameseUNet(in_channels=3, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss() # combina sigmoid + BCE, numericamente mais estável
optimizer = optim.Adam(model.parameters(), lr=0.0001)

history = train_model(model, LEVIR_train_loader, LEVIR_val_loader, criterion, optimizer, num_epochs, device=device)

# carrega o melhor modelo salvo para avaliação
checkpoint = torch.load('best_model.pt', map_location = device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval();

#### Carregamento do modelo

Carrega o melhor modelo salvo em `best_model.pt`. Execute essa célula para usar a GUI sem precisar retreinar.
> Se o kernel foi reiniciado, execute antes as células de **imports** e **arquitetura do modelo**.

In [ ]:
CARREGAR_MODELO = False # mude para True para carregar o modelo salvo

if CARREGAR_MODELO:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SiameseUNet(in_channels=3, out_channels=1).to(device)
    checkpoint = torch.load('best_model.pt', map_location = device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval();

#### Melhor modelo salvo

In [ ]:
checkpoint = torch.load('best_model.pt', map_location='cpu')

print("──" * 25)
print(" MELHOR MODELO SALVO")
print("──" * 25)
print(f"  Epoch          : {checkpoint['epoch']}")
print(f"  Validação IoU  : {checkpoint['iou']:.4f}")
print(f"  Validação Loss : {checkpoint['loss']:.4f}")
print("──" * 25)

#### Métricas da epoch com melhor IoU de validação

In [ ]:
best_epoch_idx = history['val_iou'].index(max(history['val_iou']))

print("──" * 25)
print(" MELHOR MODELO — VALIDAÇÃO")
print("──" * 25)
print(f"  Epoch      : {best_epoch_idx + 1}")
print(f"  Accuracy   : {history['val_acc'][best_epoch_idx]:.4f}")
print(f"  Precision  : {history['val_precision'][best_epoch_idx]:.4f}")
print(f"  Recall     : {history['val_recall'][best_epoch_idx]:.4f}")
print(f"  Dice       : {history['val_dice'][best_epoch_idx]:.4f}")
print("──" * 25)

#### Evolução do treinamento

Curvas de Loss, IoU e Dice por epoch — treino vs validação. Permite identificar overfitting e o ponto de convergência do modelo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Evolução do treinamento', fontsize=14)

metricas = [
    ('Loss', 'train_loss', 'val_loss'),
    ('IoU',  'train_iou',  'val_iou'),
    ('Dice', 'train_dice', 'val_dice'),
]

for ax, (titulo, treino, val) in zip(axes, metricas):
    ax.plot(history[treino], label='Treino')
    ax.plot(history[val],    label='Validação')
    ax.set_title(titulo)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_evolution.png', bbox_inches='tight')
plt.show()

#### Avaliação no conjunto de teste

Carrega o melhor modelo salvo e avalia em imagens nunca vistas durante o treinamento. Métricas registradas em `test_log.txt`.

In [ ]:
def test_model(model, test_loader, criterion, device='cuda'):
    model.to(device)
    model.eval()

    running_test_loss = 0.0
    test_iou_sum = test_acc_sum = test_precision_sum = 0
    test_recall_sum = test_dice_sum = test_samples = 0
    
    with torch.no_grad():
        for inputs1, inputs2, labels in test_loader:
            inputs1, inputs2, labels = inputs1.to(device), inputs2.to(device), labels.to(device)
            
            outputs = model(inputs1, inputs2)
            loss = criterion(outputs, labels)
            
            # acumula loss e métricas ponderadas pelo batch
            n = inputs1.size(0)
            running_test_loss += loss.item() * n
            acc, precision, recall, dice, iou = calculate_metrics(outputs, labels)
            test_acc_sum += acc * n
            test_precision_sum += precision * n
            test_recall_sum += recall * n
            test_dice_sum += dice * n
            test_iou_sum += iou * n
            test_samples += n
    
    # médias ponderadas sobre todo o conjunto de teste
    epoch_test_loss      = running_test_loss    / test_samples
    epoch_test_iou       = test_iou_sum         / test_samples
    epoch_test_acc       = test_acc_sum         / test_samples
    epoch_test_precision = test_precision_sum   / test_samples
    epoch_test_recall    = test_recall_sum      / test_samples
    epoch_test_dice      = test_dice_sum        / test_samples

    log = (f'Loss: {epoch_test_loss:.4f} | '
          f'IoU: {epoch_test_iou:.4f} | '
          f'Acc: {epoch_test_acc:.4f} | '
          f'Precision: {epoch_test_precision:.4f} | '
          f'Recall: {epoch_test_recall:.4f} | '
          f'Dice: {epoch_test_dice:.4f}')
    
    print(f'---LOG DAS MÉTRICAS NO CONJUNTO DE TESTE---')
    print(log)
    with open('test_log.txt', 'a') as f:
        f.write(log + '\n')

    return {'loss': epoch_test_loss, 'iou': epoch_test_iou, 'accuracy': epoch_test_acc, 'precision': epoch_test_precision, 
            'recall': epoch_test_recall, 'dice': epoch_test_dice}

# carrega o melhor modelo e avalia no conjunto de teste
checkpoint = torch.load('best_model.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

test_results = test_model(model, LEVIR_test_loader, criterion, device=device)

metrics = {
    'IoU': test_results['iou'],
    'Dice': test_results['dice'],
    'Accuracy': test_results['accuracy'],
    'Precision': test_results['precision'],
    'Recall': test_results['recall']
}

plt.figure(figsize=(7,4))
plt.bar(metrics.keys(), metrics.values())
plt.ylim(0,1)
plt.title('Métricas no conjunto de teste')
plt.show()

#### Visualização das predições

Exibe amostras `(pré, pós, predição, ground truth)` do conjunto de teste para avaliação visual da qualidade das máscaras geradas pelo modelo.

In [ ]:
fig, axes = plt.subplots(5, 4, figsize = (15, 15))

for i in range (5):
    # carrega par de imagens e máscara do conjunto de teste
    image1, image2, mask = LEVIR_test_dataset[i]
    
    # adiciona dimensão de batch e move para o device
    image1  = image1.unsqueeze(0).to(device)
    image2  = image2.unsqueeze(0).to(device)
    mask    = mask.unsqueeze(0).to(device)
    
    # inferência sem gradientes pois é apenas visualização
    with torch.no_grad():
        prediction = model(image1, image2)
        prediction = torch.sigmoid(prediction) 
        prediction = (prediction > 0.5).float()  # logits → máscara binária
    
    # remove dimensão de batch e move para CPU para visualização
    image1 = image1.squeeze().detach().cpu()
    image2 = image2.squeeze().detach().cpu()
    prediction = prediction.squeeze().detach().cpu()
    mask = mask.squeeze().detach().cpu()

    # pré-mudança
    axes[i, 0].imshow(image1.permute(1, 2, 0))
    axes[i, 0].set_title(f'Pré-mudança {i+1}')
    axes[i, 0].axis('off')

    # pós-mudança
    axes[i, 1].imshow(image2.permute(1, 2, 0))
    axes[i, 1].set_title(f'Pós-mudança {i+1}')
    axes[i, 1].axis('off')

    # máscara predita pelo modelo
    axes[i, 2].imshow(prediction, cmap='viridis')
    axes[i, 2].set_title(f'Predição {i+1}')
    axes[i, 2].axis('off')

    # máscara real (ground truth)
    axes[i, 3].imshow(mask, cmap='viridis')
    axes[i, 3].set_title(f'Ground Truth {i+1}')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.savefig('test_result_1.png', bbox_inches='tight')
plt.show()

In [ ]:
import random

# para visualizar em imagens aleatórias do conjunto de teste
indices = random.sample(range(len(LEVIR_test_dataset)), 5)

fig, axes = plt.subplots(5, 4, figsize = (15, 15))

for i, idx in enumerate(indices):
    # carrega par de imagens e máscara do conjunto de teste
    image1, image2, mask = LEVIR_test_dataset[idx]
    
    # adiciona dimensão de batch e move para o device
    image1  = image1.unsqueeze(0).to(device)
    image2  = image2.unsqueeze(0).to(device)
    mask    = mask.unsqueeze(0).to(device)
    
    # inferência sem gradientes pois é apenas visualização
    with torch.no_grad():
        prediction = model(image1, image2)
        prediction = torch.sigmoid(prediction) 
        prediction = (prediction > 0.5).float()  # logits → máscara binária
    
    # remove dimensão de batch e move para CPU para visualização
    image1 = image1.squeeze().detach().cpu()
    image2 = image2.squeeze().detach().cpu()
    prediction = prediction.squeeze().detach().cpu()
    mask = mask.squeeze().detach().cpu()

    # pré-mudança
    axes[i, 0].imshow(image1.permute(1, 2, 0))
    axes[i, 0].set_title(f'Pré-mudança {i+1}')
    axes[i, 0].axis('off')

    # pós-mudança
    axes[i, 1].imshow(image2.permute(1, 2, 0))
    axes[i, 1].set_title(f'Pós-mudança {i+1}')
    axes[i, 1].axis('off')

    # máscara predita pelo modelo
    axes[i, 2].imshow(prediction, cmap='viridis')
    axes[i, 2].set_title(f'Predição {i+1}')
    axes[i, 2].axis('off')

    # máscara real (ground truth)
    axes[i, 3].imshow(mask, cmap='viridis')
    axes[i, 3].set_title(f'Ground Truth {i+1}')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.savefig('test_result_2.png', bbox_inches='tight')
plt.show()

#### Interface Gradio para inferência com imagens externas

In [ ]:
import gradio as gr
import torchvision.transforms as transforms
from PIL import Image

# redimensiona para 256×256 e converte para tensor
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

def predict_change(img1, img2):
    # converte arrays numpy (Gradio) para PIL antes do transform
    t1 = transform(Image.fromarray(img1)).unsqueeze(0).to(device)
    t2 = transform(Image.fromarray(img2)).unsqueeze(0).to(device)

    # inferência: logits → sigmoid → threshold binário
    with torch.no_grad():
        output = model(t1, t2)
        pred   = (torch.sigmoid(output) > 0.5).float()

    # converte máscara para uint8 [0, 255]
    mask_256   = pred.squeeze().cpu().numpy()
    mask_uint8 = (mask_256 * 255).astype(np.uint8)

    # redimensiona máscara para o tamanho original da imagem de entrada
    orig_h, orig_w = img2.shape[:2]
    mask_resized   = cv2.resize(mask_uint8, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

    # overlay: destaca áreas de mudança em vermelho sobre a imagem pós-mudança
    red_mask = np.zeros_like(img2)
    red_mask[:, :, 0] = mask_resized
    overlay = cv2.addWeighted(img2.copy(), 0.7, red_mask, 0.5, 0)

    change_percent = float(mask_256.mean() * 100)

    return mask_resized, overlay

def clear_all():
    return None, None, None, None

# tema e estilo
theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="slate",
    font=gr.themes.GoogleFont("Inter"),
)

with gr.Blocks(title="TDE Deteccção de mudanças urbanas") as interface:

    gr.Markdown("""
    # TDE Deteccção de mudanças urbanas            
    """
    )

    clear_btn = gr.Button(
        "Limpar Tudo",
        variant="primary",
        size="sm"
    )

    with gr.Row():
        img1 = gr.Image(label="Imagem Pré-Mudança", height=300)
        img2 = gr.Image(label="Imagem Pós-Mudança", height=300)

    btn = gr.Button("Detectar Mudanças", variant="primary", size="lg")

    gr.Markdown("### Resultados")

    with gr.Row():
        out_mask    = gr.Image(label="Máscara Binária",    height=300)
        out_overlay = gr.Image(label="Overlay",            height=300)

    gr.Markdown("""
    ---
    > O modelo foi treinado no dataset **LEVIR-CD** — resultados serão 
        mais precisos com imagens de satélite RGB em resolução similar.
    """)

    btn.click(
        fn=predict_change,
        inputs=[img1, img2],
        outputs=[out_mask, out_overlay]
    )

    # botão de limpar
    clear_btn.click(
        fn=clear_all,
        outputs=[img1, img2, out_mask, out_overlay]
    )

interface.launch(share=True, theme=theme)